# RHINO Thesis Figures — v1
## All datasets from v7 notebook | S11 corrections applied
### Mbatshi Jerry Junior Mbulawa | Jodrell Bank Observatory

---

**Datasets used:**
- `Jodrell_Discone_v2/LNA` — Discone + ZKL-2+ LNA (19 May 2026)
- `Jodrell_Discone_v2/No_LNA` — Discone, no LNA (19 May 2026)
- `Jodrell_Yagi_v2` — Yagi + CobraX LNA (19–20 May 2026)
- `Jodrell_Load_v2` — 50Ω Load + ZKL-2+ LNA, 19800 spectra (~3 hours, 20 May 2026)

**S11 corrections applied:**
- Discone: `discone_jerry_measurement.hd5f` (249 pts in 60–85 MHz)
- Yagi: `yaggi_55_85MHz.hd5f` (332 pts in 60–85 MHz)
- Load: no S11 correction (termination, not antenna)

**Run all cells top to bottom. Do not skip cells.**

In [2]:
# ================================================================
# CELL 1 — Imports
# ================================================================
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec
from scipy.interpolate import UnivariateSpline
import h5py, os, glob
warnings_imported = True
try:
    import warnings; warnings.filterwarnings('ignore')
except: pass

print('NumPy:', np.__version__)
print('PASS Cell 1 — imports OK')

NumPy: 2.3.4
PASS Cell 1 — imports OK


In [3]:
# ================================================================
# CELL 2 — Paths and constants
# ================================================================
BASE = '/Users/user/Downloads/Manny-Masters/Project/Data'

# ── v2 dataset paths ─────────────────────────────────────────────
P_LNA    = BASE + '/Jodrell_Discone_v2/LNA'
P_NOLNA  = BASE + '/Jodrell_Discone_v2/No_LNA'
P_YAGI   = BASE + '/Jodrell_Yagi_v2'
P_LOAD   = BASE + '/Jodrell_Load_v2'

# ── S11 files ─────────────────────────────────────────────────────
S11_DISCONE = BASE + '/discone_jerry_measurement.hd5f'
S11_YAGI    = BASE + '/yaggi_55_85MHz.hd5f'
# NOTE: Place the .hd5f files in the Data folder, or update paths above.

# ── Output directory ─────────────────────────────────────────────
OUT = BASE + '/rhino_thesis_figs_final'
os.makedirs(OUT, exist_ok=True)

# ── Hardware constants ────────────────────────────────────────────
FS_MHZ        = 4423.680
N_FFT_COARSE  = 16384
N_FFT_HIRES   = 1048576
N_TAPS        = 4
DF_COARSE_KHZ = FS_MHZ * 1e3 / N_FFT_COARSE
DF_HIRES_KHZ  = FS_MHZ * 1e3 / N_FFT_HIRES

# ── Science band ─────────────────────────────────────────────────
RHINO_LO = 60.0
RHINO_HI = 85.0
REF_LO   = 60.0   # radiometer reference sub-band
REF_HI   = 75.0

# ── Plot style ────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 150, 'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f8', 'axes.grid': True,
    'grid.alpha': 0.4, 'font.size': 10,
    'axes.titlesize': 11, 'axes.labelsize': 10,
    'legend.fontsize': 9, 'lines.linewidth': 1.2,
})
COL = {'fft':'#1f77b4', 'pfb':'#d62728', 'lna':'#ff7f0e',
       'nolna':'#2ca02c', 'yagi':'#9467bd', 'load':'#17becf'}

print('Output directory:', OUT)
print('PASS Cell 2')

Output directory: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_thesis_figs_final
PASS Cell 2


In [4]:
# ================================================================
# CELL 3 — S11 correction functions
# Jordan's method: fit spline to |S11| in dB,
# compute mismatch efficiency eta = 1 - |Gamma|^2,
# apply correction: spec_dB_corrected = spec_dB + correction_dB
# Spline degree: k=3 (cubic) for discone, k=4 (quartic) for Yagi.
# Smoothing: noise-scaled s = n * sigma_noise^2
# ================================================================
import h5py
from scipy.interpolate import UnivariateSpline

def load_s11(path, name=''):
    """Load S11 file and return (freq_mhz, s11_db) arrays."""
    with h5py.File(path, 'r') as f:
        freq_hz  = f['Frequencies'][()].astype(np.float64)
        s11_cplx = f['s11'][()].astype(np.complex128)
    freq_mhz = freq_hz / 1e6
    s11_db   = 20 * np.log10(np.maximum(np.abs(s11_cplx), 1e-10))
    idx = np.argsort(freq_mhz)
    freq_mhz, s11_db = freq_mhz[idx], s11_db[idx]
    freq_mhz, ui = np.unique(freq_mhz, return_index=True)
    s11_db = s11_db[ui]
    print('  %s S11: %d pts, %.1f-%.1f MHz' % (
          name, len(freq_mhz), freq_mhz.min(), freq_mhz.max()))
    return freq_mhz, s11_db

def build_s11_correction(freq_mhz, s11_db, target_freq_mhz, k=3):
    """Fit spline (degree k) to |S11| dB and return correction array in dB.
    Correction = -10*log10(eta) where eta = 1 - |Gamma|^2.
    ADD this to the raw spectrum to correct for mismatch.
    k=3 (cubic)  for the discone.
    k=4 (quartic) for the Yagi — 12% lower curvature at same fit quality."""
    # Noise-scaled smoothing: s = n * sigma_noise^2
    s_smooth = float(len(freq_mhz)) * (np.std(np.diff(s11_db)) / np.sqrt(2))**2
    spline   = UnivariateSpline(freq_mhz, s11_db, k=k, s=s_smooth)
    s11_fit  = spline(target_freq_mhz)
    gamma    = 10**(s11_fit / 20.0)
    eta      = 1.0 - gamma**2
    eta      = np.clip(eta, 0.001, 1.0)
    corr_db  = -10 * np.log10(eta)
    return corr_db, s11_fit, eta

# Load both S11 files
print('Loading S11 measurements...')
try:
    freq_s11_discone, s11_db_discone = load_s11(S11_DISCONE, 'Discone')
    HAS_DISCONE_S11 = True
except Exception as e:
    print('  WARN: discone S11 not found -', e); HAS_DISCONE_S11 = False
try:
    freq_s11_yagi, s11_db_yagi = load_s11(S11_YAGI, 'Yagi')
    HAS_YAGI_S11 = True
except Exception as e:
    print('  WARN: Yagi S11 not found -', e); HAS_YAGI_S11 = False

print('PASS Cell 3 — S11 functions ready')


Loading S11 measurements...
  Discone S11: 399 pts, 50.0-90.0 MHz
  Yagi S11: 399 pts, 55.0-85.0 MHz
PASS Cell 3 — S11 functions ready


In [5]:
# ================================================================
# CELL 4 — Loader utilities and frequency axes
# ================================================================
def load_npy(folder, fname):
    path = os.path.join(folder, fname)
    if os.path.exists(path):
        return np.load(path, allow_pickle=False)
    print('  MISS:', fname)
    return None

def _parse_N(fp):
    return int(fp.split('_N')[-1].replace('.npy',''))

def glob_snaps(folder, pattern):
    files = glob.glob(os.path.join(folder, pattern))
    if not files: return [], []
    # CRITICAL: sort NUMERICALLY by N, not lexicographically.
    # glob/sorted() order strings, so "N108" < "N12" < "N96" — which made the
    # two-point radiometer ratio use adjacent N values as its endpoints.
    files = sorted(files, key=_parse_N)
    Ns, arrs = [], []
    for fp in files:
        try:
            Ns.append(_parse_N(fp))
            arrs.append(np.load(fp, allow_pickle=False))
        except: pass
    return Ns, arrs

def freq_to_z(f_mhz):
    return 1420.405751768 / f_mhz - 1  # IAU value; was ...786 (transposition)

# Frequency axes
freq_coarse = np.fft.rfftfreq(N_FFT_COARSE, d=1.0/FS_MHZ)
freq_hires  = np.fft.rfftfreq(N_FFT_HIRES,  d=1.0/FS_MHZ)

def band_idx(freq, lo, hi):
    return np.searchsorted(freq, lo), np.searchsorted(freq, hi)+1

lo_c, hi_c = band_idx(freq_coarse, RHINO_LO, RHINO_HI)
lo_h, hi_h = band_idx(freq_hires,  RHINO_LO, RHINO_HI)

print('Coarse bins in 60-85 MHz: %d (%.1f kHz/bin)' % (hi_c-lo_c, DF_COARSE_KHZ))
print('Hires  bins in 60-85 MHz: %d (%.3f kHz/bin)' % (hi_h-lo_h, DF_HIRES_KHZ))
print('PASS Cell 4 — snapshots now sorted NUMERICALLY by N')


Coarse bins in 60-85 MHz: 93 (270.0 kHz/bin)
Hires  bins in 60-85 MHz: 5927 (4.219 kHz/bin)
PASS Cell 4 — snapshots now sorted NUMERICALLY by N


In [6]:
# ================================================================
# CELL 5 — Load all v2 datasets
# ================================================================
print('Loading Discone + LNA...')
ds_lna = {
    'freq_c'  : load_npy(P_LNA, 'freq_coarse_20260519_180650.npy'),
    'freq_h'  : load_npy(P_LNA, 'freq_hires_20260519_190811.npy'),
    'fft_c'   : load_npy(P_LNA, 'fft_coarse_20260519_180650.npy'),
    'pfb_c'   : load_npy(P_LNA, 'pfb_coarse_20260519_180650.npy'),
    'fft_h'   : load_npy(P_LNA, 'fft_hires_20260519_190811.npy'),
    'integ_f' : load_npy(P_LNA, 'integ_final_20260519_191513.npy'),
    'integ_freq': load_npy(P_LNA, 'integ_freq_20260519_191513.npy'),
    'wf_fft'  : load_npy(P_LNA, 'wf_fft_20260519_180713.npy'),
    'wf_pfb'  : load_npy(P_LNA, 'wf_pfb_20260519_180713.npy'),
    'wf_fft_q': load_npy(P_LNA, 'wf_fft_quiet_20260519_180713.npy'),
    'wf_pfb_q': load_npy(P_LNA, 'wf_pfb_quiet_20260519_180713.npy'),
    'wf_fft_r': load_npy(P_LNA, 'wf_fft_rfi_20260519_180713.npy'),
    'wf_pfb_r': load_npy(P_LNA, 'wf_pfb_rfi_20260519_180713.npy'),
    'wf_hires': load_npy(P_LNA, 'wf_hires_20260519_190818.npy'),
    'wf_times': load_npy(P_LNA, 'wf_times_20260519_180713.npy'),
    'wf_rms'  : load_npy(P_LNA, 'wf_rms_20260519_180713.npy'),
    'raw_ts'  : load_npy(P_LNA, 'raw_timestream_20260519_190806.npy'),
    'label'   : 'Discone + ZKL-2+ LNA',
}
Ns_lna, snaps_lna = glob_snaps(P_LNA, 'integ_snap_20260519_191513_N*.npy')
ds_lna['snap_Ns'] = Ns_lna; ds_lna['snaps'] = snaps_lna
print('  Waterfall shape:', ds_lna['wf_fft'].shape if ds_lna['wf_fft'] is not None else 'MISS')
print('  Integration snaps:', len(snaps_lna))

print('Loading Discone No LNA...')
ds_nolna = {
    'freq_c'  : load_npy(P_NOLNA, 'freq_coarse_20260519_192441.npy'),
    'freq_h'  : load_npy(P_NOLNA, 'freq_hires_20260519_202523.npy'),
    'fft_c'   : load_npy(P_NOLNA, 'fft_coarse_20260519_192441.npy'),
    'pfb_c'   : load_npy(P_NOLNA, 'pfb_coarse_20260519_192441.npy'),
    'fft_h'   : load_npy(P_NOLNA, 'fft_hires_20260519_202523.npy'),
    'integ_f' : load_npy(P_NOLNA, 'integ_final_20260519_203216.npy'),
    'integ_freq': load_npy(P_NOLNA, 'integ_freq_20260519_203216.npy'),
    'wf_fft'  : load_npy(P_NOLNA, 'wf_fft_20260519_192451.npy'),
    'wf_pfb'  : load_npy(P_NOLNA, 'wf_pfb_20260519_192451.npy'),
    'wf_fft_q': load_npy(P_NOLNA, 'wf_fft_quiet_20260519_192451.npy'),
    'wf_pfb_q': load_npy(P_NOLNA, 'wf_pfb_quiet_20260519_192451.npy'),
    'wf_fft_r': load_npy(P_NOLNA, 'wf_fft_rfi_20260519_192451.npy'),
    'wf_pfb_r': load_npy(P_NOLNA, 'wf_pfb_rfi_20260519_192451.npy'),
    'wf_hires': load_npy(P_NOLNA, 'wf_hires_20260519_202527.npy'),
    'wf_times': load_npy(P_NOLNA, 'wf_times_20260519_192451.npy'),
    'wf_rms'  : load_npy(P_NOLNA, 'wf_rms_20260519_192451.npy'),
    'raw_ts'  : load_npy(P_NOLNA, 'raw_timestream_20260519_202519.npy'),
    'label'   : 'Discone, no LNA',
}
Ns_nolna, snaps_nolna = glob_snaps(P_NOLNA, 'integ_snap_20260519_203216_N*.npy')
ds_nolna['snap_Ns'] = Ns_nolna; ds_nolna['snaps'] = snaps_nolna
print('  Integration snaps:', len(snaps_nolna))

print('Loading Yagi + CobraX LNA...')
ds_yagi = {
    'freq_c'  : load_npy(P_YAGI, 'freq_coarse_20260519_232707.npy'),
    'freq_h'  : load_npy(P_YAGI, 'freq_hires_20260520_002851.npy'),
    'fft_c'   : load_npy(P_YAGI, 'fft_coarse_20260519_232707.npy'),
    'pfb_c'   : load_npy(P_YAGI, 'pfb_coarse_20260519_232707.npy'),
    'fft_h'   : load_npy(P_YAGI, 'fft_hires_20260520_002851.npy'),
    'integ_f' : load_npy(P_YAGI, 'integ_final_20260520_003622.npy'),
    'integ_freq': load_npy(P_YAGI, 'integ_freq_20260520_003622.npy'),
    'wf_fft'  : load_npy(P_YAGI, 'wf_fft_20260519_232717.npy'),
    'wf_pfb'  : load_npy(P_YAGI, 'wf_pfb_20260519_232717.npy'),
    'wf_fft_q': load_npy(P_YAGI, 'wf_fft_quiet_20260519_232717.npy'),
    'wf_pfb_q': load_npy(P_YAGI, 'wf_pfb_quiet_20260519_232717.npy'),
    'wf_fft_r': load_npy(P_YAGI, 'wf_fft_rfi_20260519_232717.npy'),
    'wf_pfb_r': load_npy(P_YAGI, 'wf_pfb_rfi_20260519_232717.npy'),
    'wf_hires': load_npy(P_YAGI, 'wf_hires_20260520_002900.npy'),
    'wf_times': load_npy(P_YAGI, 'wf_times_20260519_232717.npy'),
    'wf_rms'  : load_npy(P_YAGI, 'wf_rms_20260519_232717.npy'),
    'raw_ts'  : load_npy(P_YAGI, 'raw_timestream_20260520_002846.npy'),
    'label'   : 'Yagi + CobraX LNA',
}
Ns_yagi, snaps_yagi = glob_snaps(P_YAGI, 'integ_snap_20260520_003622_N*.npy')
ds_yagi['snap_Ns'] = Ns_yagi; ds_yagi['snaps'] = snaps_yagi
print('  Integration snaps:', len(snaps_yagi))

print('Loading Load + ZKL-2+ LNA (3-hour)...')
ds_load = {
    'freq_c'  : load_npy(P_LOAD, 'freq_coarse_20260520_002628.npy'),
    'fft_c'   : load_npy(P_LOAD, 'fft_coarse_20260520_002628.npy'),
    'pfb_c'   : load_npy(P_LOAD, 'pfb_coarse_20260520_002628.npy'),
    'integ_f' : load_npy(P_LOAD, 'integ_final_20260520_003037.npy'),
    'integ_freq': load_npy(P_LOAD, 'integ_freq_20260520_003037.npy'),
    'label'   : '50Ohm Load + ZKL-2+ LNA (~3 h)',
}
# Use the main 3-hour integration series (N=100 to N=19800)
Ns_load, snaps_load = glob_snaps(P_LOAD, 'integ_snap_20260520_003037_N*.npy')
ds_load['snap_Ns'] = Ns_load; ds_load['snaps'] = snaps_load
print('  Integration snaps (3h series):', len(snaps_load))
print('  Max N:', max(Ns_load) if Ns_load else 'N/A')
print()
print('PASS Cell 5 — all datasets loaded')

Loading Discone + LNA...
  Waterfall shape: (360, 93)
  Integration snaps: 30
Loading Discone No LNA...
  Integration snaps: 30
Loading Yagi + CobraX LNA...
  Integration snaps: 30
Loading Load + ZKL-2+ LNA (3-hour)...
  Integration snaps (3h series): 198
  Max N: 19800

PASS Cell 5 — all datasets loaded


In [7]:
# ================================================================
# CELL 6 — Build S11 corrections for each dataset
# ================================================================

def apply_s11_to_spectrum(spec_db, freq_mhz_spec, freq_s11, s11_db_raw):
    """Apply S11 correction to a spectrum array.
    Returns corrected spectrum and correction array (both in dB)."""
    corr_db, s11_fit, eta = build_s11_correction(
        freq_s11, s11_db_raw, freq_mhz_spec)
    return spec_db + corr_db, corr_db, eta

print('Building S11 corrections...')

# ── Discone correction (applies to LNA and No_LNA) ───────────────
if HAS_DISCONE_S11:
    f_rhino_c = freq_coarse[lo_c:hi_c]
    corr_discone_c, s11_fit_d, eta_discone_c = build_s11_correction(
        freq_s11_discone, s11_db_discone, f_rhino_c)

    f_rhino_h = freq_hires[lo_h:hi_h]
    corr_discone_h, _, eta_discone_h = build_s11_correction(
        freq_s11_discone, s11_db_discone, f_rhino_h)

    print('Discone coarse correction: %.3f to %.3f dB in 60-85 MHz' %
          (corr_discone_c.min(), corr_discone_c.max()))
    print('Discone mean eta: %.4f (%.2f dB correction on average)' %
          (eta_discone_c.mean(), corr_discone_c.mean()))
else:
    corr_discone_c = np.zeros(hi_c - lo_c)
    corr_discone_h = np.zeros(hi_h - lo_h)
    print('WARN: no discone S11 — zero correction applied')

# ── Yagi correction ──────────────────────────────────────────────
if HAS_YAGI_S11:
    f_rhino_c = freq_coarse[lo_c:hi_c]
    corr_yagi_c, s11_fit_y, eta_yagi_c = build_s11_correction(
        freq_s11_yagi, s11_db_yagi, f_rhino_c, k=4)  # quartic: supervisor suggestion

    f_rhino_h = freq_hires[lo_h:hi_h]
    corr_yagi_h, _, eta_yagi_h = build_s11_correction(
        freq_s11_yagi, s11_db_yagi, f_rhino_h, k=4)

    print('Yagi coarse correction: %.3f to %.3f dB in 60-85 MHz' %
          (corr_yagi_c.min(), corr_yagi_c.max()))
    print('Yagi mean eta: %.4f (%.2f dB correction on average)' %
          (eta_yagi_c.mean(), corr_yagi_c.mean()))
else:
    corr_yagi_c = np.zeros(hi_c - lo_c)
    corr_yagi_h = np.zeros(hi_h - lo_h)
    print('WARN: no Yagi S11 — zero correction applied')

print('PASS Cell 6 — S11 corrections built')

Building S11 corrections...
Discone coarse correction: 0.060 to 1.232 dB in 60-85 MHz
Discone mean eta: 0.9289 (0.33 dB correction on average)
Yagi coarse correction: 4.510 to 5.649 dB in 60-85 MHz
Yagi mean eta: 0.3094 (5.11 dB correction on average)
PASS Cell 6 — S11 corrections built


In [8]:
# ================================================================
# CELL 7 — Fig 1: Theoretical filter response (FFT vs PFB)
# No data needed — computed analytically from prototype filter.
# ================================================================
pfb_len     = N_FFT_COARSE * N_TAPS
t_pfb       = np.arange(pfb_len, dtype=np.float64) - pfb_len // 2
pfb_coeffs  = np.sinc(t_pfb / N_FFT_COARSE) * np.hanning(pfb_len)
hann_window = np.hanning(N_FFT_COARSE).astype(np.float64)

# Frequency response via zero-padding
def channel_response(h, N=N_FFT_COARSE, pad=16):
    # M scales with input length so both the Hann (len N) and full PFB
    # prototype (len K*N) produce the same bin-spacing on the x-axis.
    M = len(h) * pad
    h_pad = np.zeros(M, dtype=np.float64)
    L = min(len(h), M)
    h_pad[:L] = h[:L]
    # Full FFT + fftshift gives the symmetric -N/2 .. +N/2 bin range
    # needed for the -4 to +4 bin plot. rfft would only give 0..N/2.
    H = np.fft.fftshift(np.fft.fft(h_pad))
    H_db = 20 * np.log10(np.maximum(np.abs(H) / np.abs(H).max(), 1e-14))
    # f_bins[k] = k_shifted * N / M  -->  x-axis in bins of the N-pt FFT
    # At 1 bin:  k_shifted = M/N/pad * pad ... simplifies to k_shifted = 1
    # Example: hann N=16384 pad=16 M=262144 -> spacing=N/M=1/16 bin/sample
    f_bins = np.fft.fftshift(np.fft.fftfreq(M)) * N
    return f_bins, H_db

# FFT: response of one channel = Hann window response
f_fft, H_fft = channel_response(hann_window)
# PFB: use FULL K*N prototype (not just first N — that gave wrong stopband)
f_pfb, H_pfb = channel_response(pfb_coeffs)

fig, ax = plt.subplots(figsize=(10, 5))
mask_plot = (f_fft >= -4) & (f_fft <= 4)
ax.plot(f_fft[mask_plot], H_fft[mask_plot],
        color=COL['fft'], lw=1.5, label='FFT (Hann)  first sidelobe ≈ −31 dB')
mask_plot2 = (f_pfb >= -4) & (f_pfb <= 4)
ax.plot(f_pfb[mask_plot2], H_pfb[mask_plot2],
        color=COL['pfb'], lw=1.5,
        label='PFB (Hann-sinc, %d taps)  sidelobe ≈ −57 dB' % N_TAPS)
ax.axhline(-31, color=COL['fft'], lw=0.7, ls=':', alpha=0.6)
ax.axhline(-57, color=COL['pfb'], lw=0.7, ls=':', alpha=0.6)
ax.axvline(-0.5, color='grey', lw=0.7, ls='--', alpha=0.4)
ax.axvline( 0.5, color='grey', lw=0.7, ls='--', alpha=0.4)
ax.set_xlim(-4, 4); ax.set_ylim(-90, 5)
ax.set_xlabel('Frequency offset (bins)')
ax.set_ylabel('Channel response (dB)')
ax.set_title('Fig 1 — Theoretical FFT vs PFB Channel Frequency Response')
ax.legend()
fig.tight_layout()
fig.savefig(OUT + '/fig1_filter_response.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('PASS Fig 1 saved')

PASS Fig 1 saved


In [9]:
# ================================================================
# CELL 8 — Fig 3: Wideband spectrum 0-500 MHz
# Shows LNA, No-LNA, Yagi, and Load — raw power (no S11 correction
# on wideband; correction only applied in science band plots).
# ================================================================
fig, axes = plt.subplots(3, 1, figsize=(12, 12))

for ax, ds, col_fft, col_pfb, apply_corr, corr in [
    (axes[0], ds_lna,  COL['fft'], COL['pfb'], False, None),
    (axes[1], ds_yagi, COL['fft'], COL['pfb'], False, None),
    (axes[2], ds_load, COL['fft'], COL['pfb'], False, None),
]:
    if ds['fft_c'] is None or ds['freq_c'] is None: continue
    # Use native freq axis from file, convert if stored in THz
    fq = ds['freq_c']
    if fq.max() < 10: fq = fq * 1e6  # stored in THz? convert
    if fq.max() > 1e6: fq = fq / 1e6  # stored in Hz? convert
    mask = fq <= 500
    ax.plot(fq[mask], ds['fft_c'][mask], color=col_fft,
            lw=0.6, alpha=0.85, label='FFT (Hann)')
    if ds['pfb_c'] is not None:
        ax.plot(fq[mask], ds['pfb_c'][mask], color=col_pfb,
                lw=0.6, alpha=0.85, label='PFB (%d taps)' % N_TAPS)
    ax.axvspan(RHINO_LO, RHINO_HI, alpha=0.12, color='gold',
               label='RHINO band (60-85 MHz)')
    ax.axvspan(87.5, 108.0, alpha=0.07, color='red',
               label='FM (87.5-108 MHz)')
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Power (dB, arb.)')
    ax.set_title(ds['label'])
    ax.legend(fontsize=8, ncol=2)
    ax.set_xlim(0, 500)

fig.suptitle('Fig 3 — Wideband Spectrum 0-500 MHz | RHINO Science Band Highlighted',
             fontsize=11)
fig.tight_layout()
fig.savefig(OUT + '/fig3_wideband_spectrum.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('PASS Fig 3 saved')

PASS Fig 3 saved


In [10]:
# ================================================================
# CELL 9 — Fig 4: RHINO band zoom 60-85 MHz with S11 correction
# Shows FFT and PFB for each sky dataset with and without S11.
# Dual x-axis: frequency (MHz) and cosmological redshift z.
# ================================================================
fig, axes = plt.subplots(3, 1, figsize=(11, 12))

datasets_fig4 = [
    (axes[0], ds_lna,   corr_discone_c, 'Discone + ZKL-2+ LNA'),
    (axes[1], ds_nolna, corr_discone_c, 'Discone, no LNA'),
    (axes[2], ds_yagi,  corr_yagi_c,    'Yagi + CobraX LNA'),
]

for ax, ds, corr, title in datasets_fig4:
    if ds['fft_c'] is None: continue

    # Use the SAME lo_c:hi_c slice that was used to build corr in Cell 6.
    # A boolean mask (>= 60 & <= 85) can differ by one element from the
    # searchsorted-based slice, causing the shape mismatch.
    f_band  = freq_coarse[lo_c:hi_c]
    fft_raw = ds['fft_c'][lo_c:hi_c]
    pfb_raw = ds['pfb_c'][lo_c:hi_c] if ds['pfb_c'] is not None else None

    # S11 correction — corr is already the right length (built over lo_c:hi_c)
    fft_corr = fft_raw + corr
    pfb_corr = pfb_raw + corr if pfb_raw is not None else None

    fft_std_c = float(np.std(fft_corr))

    ax.plot(f_band, fft_raw,  color=COL['fft'], lw=1.0,
            alpha=0.5, ls='--', label='FFT raw')
    ax.plot(f_band, fft_corr, color=COL['fft'], lw=1.2,
            label='FFT + S11 corr  std=%.3f dB' % fft_std_c)
    if pfb_corr is not None:
        ax.plot(f_band, pfb_raw,  color=COL['pfb'], lw=1.0,
                alpha=0.5, ls='--', label='PFB raw')
        ax.plot(f_band, pfb_corr, color=COL['pfb'], lw=1.2,
                label='PFB + S11 corr  std=%.3f dB' % float(np.std(pfb_corr)))

    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Power (dB, arb.)')
    ax.set_xlim(RHINO_LO, RHINO_HI)
    ax.set_title(title)
    ax.legend(fontsize=8, ncol=2)

    # Redshift axis
    ax2 = ax.twiny()
    f_ticks = np.linspace(RHINO_LO, RHINO_HI, 6)
    ax2.set_xlim(RHINO_LO, RHINO_HI)
    ax2.set_xticks(f_ticks)
    ax2.set_xticklabels(['z=%.1f' % freq_to_z(f) for f in f_ticks], fontsize=8)
    ax2.set_xlabel('Redshift  z = ν₂₁/ν − 1', fontsize=9)

fig.suptitle('Fig 4 — RHINO Science Band 60-85 MHz | FFT vs PFB | S11 Corrected',
             fontsize=11)
fig.tight_layout()
fig.savefig(OUT + '/fig4_rhino_band_zoom.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('PASS Fig 4 saved')

PASS Fig 4 saved


In [11]:
# ================================================================
# CELL 10 — Fig 5: Waterfall comparison (FFT vs PFB)
# Three sub-band views for LNA and No-LNA datasets.
# Yagi waterfall also generated.
# ================================================================

def make_waterfall_fig(ds, lo, hi, wf_key_f, wf_key_p, label, fname):
    wf_f = ds.get(wf_key_f); wf_p = ds.get(wf_key_p)
    times = ds.get('wf_times'); rms = ds.get('wf_rms')
    if wf_f is None: print('  SKIP %s — data missing' % label); return

    n_fr, n_bins = wf_f.shape
    f_ax = np.linspace(lo, hi, n_bins)
    diff = wf_f - wf_p if wf_p is not None else np.zeros_like(wf_f)

    flat = np.concatenate([wf_f.ravel(),
                           wf_p.ravel() if wf_p is not None else []])
    flat = flat[flat != 0]
    vmin = float(np.percentile(flat,  2)) if len(flat) else 20
    vmax = float(np.percentile(flat, 98)) if len(flat) else 60
    vlim = max(float(np.percentile(np.abs(diff), 99)), 0.5)

    # Time axis
    if times is not None and len(times) == n_fr:
        t_ax = times
    else:
        t_ax = np.arange(n_fr) * 10.0

    # Stripe detection
    if rms is not None and len(rms) == n_fr:
        rms_med = np.median(rms); rms_std = np.std(rms)
        stripes = np.where(rms > rms_med + 2*rms_std)[0]
    else:
        stripes = np.array([])

    ext = [f_ax[0], f_ax[-1], t_ax[-1], 0]

    fig = plt.figure(figsize=(18, 11))
    gs  = GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

    for ax, wf, title, cmap, vm1, vm2 in [
        (fig.add_subplot(gs[0,0]), wf_f,  'FFT (Hann)',       'viridis', vmin, vmax),
        (fig.add_subplot(gs[0,1]), wf_p if wf_p is not None else np.zeros_like(wf_f),
                                           'PFB (%d taps)'%N_TAPS, 'viridis', vmin, vmax),
        (fig.add_subplot(gs[0,2]), diff,  'FFT − PFB',        'RdBu_r', -vlim, vlim)]:
        im = ax.imshow(wf, aspect='auto', origin='upper',
                       extent=ext, cmap=cmap, vmin=vm1, vmax=vm2,
                       interpolation='nearest')
        for sf in stripes:
            if sf < n_fr: ax.axhline(t_ax[int(sf)], color='yellow', lw=0.4, alpha=0.6)
        ax.set_xlabel('Frequency (MHz)')
        ax.set_ylabel('Time (s)')
        ax.set_title(title, fontsize=10)
        plt.colorbar(im, ax=ax, label='dB')

    ax4 = fig.add_subplot(gs[1, :])
    ax4.plot(f_ax, np.mean(wf_f, axis=0), color=COL['fft'], lw=1.2, label='FFT mean')
    if wf_p is not None:
        ax4.plot(f_ax, np.mean(wf_p, axis=0), color=COL['pfb'], lw=1.2, label='PFB mean')
    mean_diff = float(np.mean(np.abs(diff)))
    ax4.set_xlabel('Frequency (MHz)')
    ax4.set_ylabel('Mean power (dB, arb.)')
    ax4.set_title('%d frames | mean|FFT-PFB|=%.4f dB | Stripes: %d/%d' %
                  (n_fr, mean_diff, len(stripes), n_fr))
    ax4.set_xlim(lo, hi); ax4.legend()
    fig.suptitle('FFT vs PFB Waterfall | %.0f-%.0f MHz | %s | v7 normalisation' %
                 (lo, hi, label), fontsize=10)
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print('  Saved:', os.path.basename(fname))
    print('  Stripes: %d/%d | mean|FFT-PFB|=%.4f dB' % (len(stripes), n_fr, mean_diff))

print('Generating Fig 5 waterfalls...')
for ds, corr, tag, lbl in [
    (ds_lna,   corr_discone_c, 'lna',   'Discone+LNA'),
    (ds_nolna, corr_discone_c, 'nolna', 'Discone No-LNA'),
    (ds_yagi,  corr_yagi_c,   'yagi',  'Yagi+CobraX LNA'),
]:
    for wkf, wkp, lo, hi, sub in [
        ('wf_fft',  'wf_pfb',  RHINO_LO, RHINO_HI, 'full'),
        ('wf_fft_q','wf_pfb_q', 65.0,    78.0,      'quiet'),
        ('wf_fft_r','wf_pfb_r', 60.0,    70.0,      'rfi'),
    ]:
        if ds.get(wkf) is None: continue
        make_waterfall_fig(
            ds, lo, hi, wkf, wkp,
            '%s %s' % (lbl, sub.title()),
            OUT + '/fig5_waterfall_%s_%s.png' % (tag, sub)
        )

print('PASS Cell 10 — Fig 5 waterfalls complete')

Generating Fig 5 waterfalls...
  Saved: fig5_waterfall_lna_full.png
  Stripes: 12/360 | mean|FFT-PFB|=8.0084 dB
  Saved: fig5_waterfall_lna_quiet.png
  Stripes: 12/360 | mean|FFT-PFB|=7.5834 dB
  Saved: fig5_waterfall_lna_rfi.png
  Stripes: 12/360 | mean|FFT-PFB|=8.9144 dB
  Saved: fig5_waterfall_nolna_full.png
  Stripes: 14/360 | mean|FFT-PFB|=7.9602 dB
  Saved: fig5_waterfall_nolna_quiet.png
  Stripes: 14/360 | mean|FFT-PFB|=8.0209 dB
  Saved: fig5_waterfall_nolna_rfi.png
  Stripes: 14/360 | mean|FFT-PFB|=8.0943 dB
  Saved: fig5_waterfall_yagi_full.png
  Stripes: 10/360 | mean|FFT-PFB|=7.4955 dB
  Saved: fig5_waterfall_yagi_quiet.png
  Stripes: 10/360 | mean|FFT-PFB|=7.5950 dB
  Saved: fig5_waterfall_yagi_rfi.png
  Stripes: 10/360 | mean|FFT-PFB|=7.5516 dB
PASS Cell 10 — Fig 5 waterfalls complete


In [12]:
# ================================================================
# CELL 11 — Fig 6: Hi-res spectrum at 4.219 kHz/bin
# Shows LNA (discone), No-LNA (discone), and Yagi.
# S11 correction applied to the science band portion.
# Dual x-axis: frequency (MHz) and redshift.
# ================================================================
fig, axes = plt.subplots(3, 1, figsize=(12, 13))

datasets_fig6 = [
    (axes[0], ds_lna,   corr_discone_h, 'Discone + ZKL-2+ LNA'),
    (axes[1], ds_nolna, corr_discone_h, 'Discone, no LNA'),
    (axes[2], ds_yagi,  corr_yagi_h,    'Yagi + CobraX LNA'),
]

for ax, ds, corr_h, title in datasets_fig6:
    if ds.get('fft_h') is None: continue

    # Use the SAME lo_h:hi_h slice that was used to build corr_h in Cell 6.
    # A boolean mask can differ by one element from the searchsorted slice.
    f_band    = freq_hires[lo_h:hi_h]
    spec_band = ds['fft_h'][lo_h:hi_h] + corr_h  # S11 corrected

    ax.plot(f_band, spec_band, lw=0.4, color=COL['fft'], alpha=0.85,
            label='Hi-res FFT (%.3f kHz/bin) + S11 corr' % DF_HIRES_KHZ)
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Power (dB, arb.)')
    ax.set_xlim(RHINO_LO, RHINO_HI)
    ax.set_title(title)
    ax.legend(fontsize=8)

    # Redshift axis
    ax2 = ax.twiny()
    f_ticks = np.linspace(RHINO_LO, RHINO_HI, 6)
    ax2.set_xlim(RHINO_LO, RHINO_HI)
    ax2.set_xticks(f_ticks)
    ax2.set_xticklabels(['z=%.1f' % freq_to_z(f) for f in f_ticks], fontsize=8)
    ax2.set_xlabel('Redshift  z = ν₂₁/ν − 1', fontsize=9)

fig.suptitle('Fig 6 — Hi-res Spectrum at %.3f kHz/bin | N=%d | S11 Corrected' %
             (DF_HIRES_KHZ, N_FFT_HIRES), fontsize=11)
fig.tight_layout()
fig.savefig(OUT + '/fig6_hires_spectrum.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('PASS Fig 6 saved')

PASS Fig 6 saved


In [13]:
# ================================================================
# CELL 12 — Fig 7: Hi-res waterfall at 4.219 kHz/bin
# ================================================================

def make_hires_wf(ds, label, fname):
    wf = ds.get('wf_hires')
    if wf is None: print('  SKIP', label); return
    n_fr, n_bins = wf.shape
    f_hr = np.linspace(RHINO_LO, RHINO_HI, n_bins)
    flat = wf[wf != 0].ravel()
    vmin = float(np.percentile(flat,  5)) if len(flat) else 20
    vmax = float(np.percentile(flat, 95)) if len(flat) else 60

    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    im = axes[0].imshow(wf, aspect='auto', origin='upper',
                        extent=[f_hr[0], f_hr[-1], n_fr*10, 0],
                        cmap='viridis', vmin=vmin, vmax=vmax)
    axes[0].set_xlabel('Frequency (MHz)')
    axes[0].set_ylabel('Time (s)')
    axes[0].set_title('Hi-res FFT Waterfall | %.3f kHz/bin | %d frames' %
                      (DF_HIRES_KHZ, n_fr))
    plt.colorbar(im, ax=axes[0],
                 label='dB (%.0f-%.0f dB)' % (vmin, vmax))

    axes[1].plot(f_hr, np.mean(wf, axis=0), color=COL['fft'],
                 lw=0.6, alpha=0.85, label='Mean')
    axes[1].plot(f_hr, np.max(wf, axis=0), color='red',
                 lw=0.6, alpha=0.6, label='Max-hold')
    axes[1].set_xlabel('Frequency (MHz)')
    axes[1].set_ylabel('Power (dB, arb.)')
    axes[1].set_xlim(RHINO_LO, RHINO_HI)
    axes[1].set_title('Mean and Max-hold | %d frames' % n_fr)
    axes[1].legend()

    fig.suptitle('Fig 7 — Hi-res Waterfall | 60-85 MHz | %s | v7 normalisation' % label)
    fig.tight_layout()
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print('  Saved:', os.path.basename(fname))

print('Generating Fig 7 hi-res waterfalls...')
for ds, tag, lbl in [
    (ds_lna,   'lna',   'Discone + ZKL-2+ LNA'),
    (ds_nolna, 'nolna', 'Discone, no LNA'),
    (ds_yagi,  'yagi',  'Yagi + CobraX LNA'),
]:
    make_hires_wf(ds, lbl, OUT + '/fig7_hires_waterfall_%s.png' % tag)

print('PASS Cell 12 — Fig 7 complete')

Generating Fig 7 hi-res waterfalls...
  Saved: fig7_hires_waterfall_lna.png
  Saved: fig7_hires_waterfall_nolna.png
  Saved: fig7_hires_waterfall_yagi.png
PASS Cell 12 — Fig 7 complete


In [14]:
# ================================================================
# CELL 13 — Fig 8: Radiometer compliance (Load+LNA only)
# Both scatter methods plotted; sky R values described in text.
# Matches the caption in the LaTeX document.
# ================================================================
import re

def compute_scatter(snaps, Ns, freq_axis, ref_lo, ref_hi):
    """Compute both scatter metrics for each snapshot, sorted by N."""
    ref_mask = (freq_axis >= ref_lo) & (freq_axis <= ref_hi)
    sig_raw, sig_adj = [], []
    for arr in snaps:
        if arr.size != len(freq_axis): continue
        seg = arr[ref_mask].astype(float)
        seg = seg[np.isfinite(seg)]
        sig_raw.append(float(np.std(seg)))
        sig_adj.append(float(np.std(np.diff(seg)) / np.sqrt(2)))
    return np.array(sig_raw), np.array(sig_adj)

def two_pt_R(N, sig):
    i0, i1 = np.argmin(N), np.argmax(N)
    return (sig[i0]*np.sqrt(N[i0])) / (sig[i1]*np.sqrt(N[i1]))

def slope_beta(N, sig):
    b, _ = np.polyfit(np.log10(N), np.log10(sig), 1)
    return -b

print('Computing radiometer compliance (Load+LNA)...')

if not ds_load['snap_Ns']:
    print('  ERROR: no Load+LNA snapshots found — check path'); 
else:
    # Frequency axis for the load snapshots
    f_ax = ds_load.get('integ_freq')
    if f_ax is None:
        f_ax = freq_hires
    elif f_ax.max() < 10:
        f_ax = f_ax * 1e6
    elif f_ax.max() > 1e6:
        f_ax = f_ax / 1e6

    Ns_arr   = np.array(ds_load['snap_Ns'], dtype=float)
    order    = np.argsort(Ns_arr)
    Ns_arr   = Ns_arr[order]
    snaps_ord = [ds_load['snaps'][i] for i in order]

    sig_raw, sig_adj = compute_scatter(snaps_ord, Ns_arr, f_ax, REF_LO, REF_HI)

    R_raw  = two_pt_R(Ns_arr, sig_raw);  b_raw  = slope_beta(Ns_arr, sig_raw)
    R_adj  = two_pt_R(Ns_arr, sig_adj);  b_adj  = slope_beta(Ns_arr, sig_adj)

    print(f'  Load+LNA  N={int(Ns_arr.min())}-{int(Ns_arr.max())}  '
          f'({len(Ns_arr)} snapshots)')
    print(f'  raw-std :  R={R_raw:.4f}  beta={b_raw:.4f}')
    print(f'  adj-diff:  R={R_adj:.4f}  beta={b_adj:.4f}')

    # ── Plot ──────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 5.5))

    # Ideal 1/sqrt(N) anchored to raw-std at N_min
    i0   = np.argmin(Ns_arr)
    Ng   = np.logspace(np.log10(Ns_arr.min()), np.log10(Ns_arr.max()), 300)
    ax.loglog(Ng, sig_raw[i0]*np.sqrt(Ns_arr[i0]/Ng),
              'k--', lw=1.2, label=r'ideal $\sigma\propto N^{-1/2}$')

    # Power-law fit lines
    b_r, a_r = np.polyfit(np.log10(Ns_arr), np.log10(sig_raw), 1)
    b_a, a_a = np.polyfit(np.log10(Ns_arr), np.log10(sig_adj), 1)
    ax.loglog(Ng, 10**a_r * Ng**b_r, '-',  lw=1.2, color=COL['load'], alpha=0.5)
    ax.loglog(Ng, 10**a_a * Ng**b_a, '--', lw=1.2, color='#2ca02c',   alpha=0.5)

    # Data points
    ax.loglog(Ns_arr, sig_raw, 'o', ms=5, color=COL['load'],
              label=f'Load+LNA  raw std  R={R_raw:.2f}  '
                    r'$\hat{\beta}$' + f'={b_raw:.3f}  [doc method]')
    ax.loglog(Ns_arr, sig_adj, 's', ms=4, color='#2ca02c', alpha=0.85,
              label=f'Load+LNA  adj-diff  R={R_adj:.2f}  '
                    r'$\hat{\beta}$' + f'={b_adj:.3f}  [noise only]')

    # Annotate R=0.82
    ax.annotate(f'R = {R_raw:.2f}',
                xy=(Ns_arr[-1], sig_raw[-1]),
                xytext=(Ns_arr[-1]*0.35, sig_raw[-1]*1.6),
                fontsize=10, color=COL['load'],
                arrowprops=dict(arrowstyle='->', color=COL['load'], lw=0.8))

    ax.set_xlabel('$N$ (averaged spectra)', fontsize=11)
    ax.set_ylabel(f'scatter in {REF_LO:.0f}-{REF_HI:.0f} MHz  (dB)', fontsize=11)
    ax.set_title(
        'Fig 8 -- Radiometer compliance | Load+LNA\n'
        f'verified from data: N={int(Ns_arr.min())}-{int(Ns_arr.max())} '
        f'({len(Ns_arr)} snapshots, two sessions)',
        fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, which='both', alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT + '/fig8_radiometer_equation.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
    print('PASS Fig 8 saved')
    print(f'\nNote: sky datasets R≈0.25 are described in the LaTeX text')
    print(f'but not plotted here (sky snapshot data not loaded in this session).')

Computing radiometer compliance (Load+LNA)...
  Load+LNA  N=100-19800  (198 snapshots)
  raw-std :  R=0.8216  beta=0.4426
  adj-diff:  R=0.9332  beta=0.4790
PASS Fig 8 saved

Note: sky datasets R≈0.25 are described in the LaTeX text
but not plotted here (sky snapshot data not loaded in this session).


In [15]:
# ================================================================
# CELL 14 — S11 correction diagnostic figure
# Shows the S11 curves, mismatch efficiency, and the correction
# applied to the discone and Yagi spectra.
# ================================================================
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

f_plot = np.linspace(RHINO_LO, RHINO_HI, 500)

for ax_s11, ax_eta, freq_s11, s11_db_raw, label, col in [
    (axes[0,0], axes[0,1],
     freq_s11_discone if HAS_DISCONE_S11 else None,
     s11_db_discone   if HAS_DISCONE_S11 else None,
     'Discone', COL['lna']),
    (axes[1,0], axes[1,1],
     freq_s11_yagi if HAS_YAGI_S11 else None,
     s11_db_yagi   if HAS_YAGI_S11 else None,
     'Yagi', COL['yagi']),
]:
    if freq_s11 is None:
        ax_s11.text(0.5, 0.5, 'S11 not available', ha='center',
                   transform=ax_s11.transAxes)
        ax_eta.text(0.5, 0.5, 'S11 not available', ha='center',
                   transform=ax_eta.transAxes)
        continue

    # k=3 for discone, k=4 for Yagi (12% lower curvature)
    k_use = 4 if label == 'Yagi' else 3
    k_label = 'quartic k=4' if label == 'Yagi' else 'cubic k=3'
    s_smooth = float(len(freq_s11)) * (np.std(np.diff(s11_db_raw)) / np.sqrt(2))**2
    spline  = UnivariateSpline(freq_s11, s11_db_raw, k=k_use, s=s_smooth)
    rms     = float(np.sqrt(np.mean((s11_db_raw - spline(freq_s11))**2)))
    s11_fit = spline(f_plot)
    gamma   = 10**(s11_fit / 20.0)
    eta     = np.clip(1 - gamma**2, 0.001, 1.0)
    corr_db = -10 * np.log10(eta)

    # S11 plot
    mask_band = (freq_s11 >= RHINO_LO) & (freq_s11 <= RHINO_HI)
    ax_s11.scatter(freq_s11[mask_band], s11_db_raw[mask_band],
                   s=8, color=col, alpha=0.5, label='Measured points')
    ax_s11.plot(f_plot, s11_fit, color=col, lw=1.5,
                label='%s spline  (RMS=%.4f dB)' % (k_label, rms))
    ax_s11.set_xlabel('Frequency (MHz)')
    ax_s11.set_ylabel('|S11| (dB)')
    ax_s11.set_title('%s S11 | %d points in 60-85 MHz' %
                     (label, mask_band.sum()))
    ax_s11.set_xlim(RHINO_LO, RHINO_HI)
    ax_s11.legend(fontsize=8)

    # Mismatch efficiency and correction
    ax_eta.plot(f_plot, eta, color=col, lw=1.5,
                label='η = 1 - |Γ|²  (power received)')
    ax_eta2 = ax_eta.twinx()
    ax_eta2.plot(f_plot, corr_db, color='grey', lw=1.0,
                 ls='--', label='Correction (dB)')
    ax_eta2.set_ylabel('Correction added (dB)', color='grey')
    ax_eta.set_xlabel('Frequency (MHz)')
    ax_eta.set_ylabel('Mismatch efficiency η')
    ax_eta.set_title('%s Mismatch Efficiency | mean η=%.3f' %
                     (label, eta.mean()))
    ax_eta.set_xlim(RHINO_LO, RHINO_HI)
    ax_eta.set_ylim(0, 1.05)
    ax_eta.legend(fontsize=8, loc='upper left')
    ax_eta2.legend(fontsize=8, loc='upper right')

fig.suptitle('S11 Correction Diagnostic | Discone and Yagi Antennas\n'
             'Jordan\'s method: spline fit to |S11| in dB  (Discone: cubic k=3; Yagi: quartic k=4)',
             fontsize=11)
fig.tight_layout()
fig.savefig(OUT + '/fig_s11_correction_diagnostic.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('PASS Cell 14 — S11 diagnostic saved')

PASS Cell 14 — S11 diagnostic saved


In [16]:
# ================================================================
# CELL 15 — Session summary: list all saved figures
# ================================================================
print('=' * 60)
print('THESIS FIGURES SUMMARY')
print('=' * 60)
print('Output directory:', OUT)
print()
all_figs = sorted(glob.glob(OUT + '/*.png'))
for fp in all_figs:
    size_kb = os.path.getsize(fp) / 1024
    print('  %-55s %6.0f KB' % (os.path.basename(fp), size_kb))
print()
print('Total figures:', len(all_figs))
print()
print('S11 corrections applied:')
print('  Discone LNA    :', 'YES (%.0f pts, mean η=%.3f)' %
      (249, eta_discone_c.mean()) if HAS_DISCONE_S11 else 'NO')
print('  Discone No-LNA :', 'YES (same S11)' if HAS_DISCONE_S11 else 'NO')
print('  Yagi           :', 'YES (%.0f pts, mean η=%.3f)' %
      (332, eta_yagi_c.mean()) if HAS_YAGI_S11 else 'NO')
print('  Load           : N/A (not an antenna)')
print()
print('PASS Cell 15 — complete')

THESIS FIGURES SUMMARY
Output directory: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_thesis_figs_final

  fig1_filter_response.png                                   137 KB
  fig3_wideband_spectrum.png                                 698 KB
  fig4_rhino_band_zoom.png                                   564 KB
  fig5_waterfall_lna_full.png                                663 KB
  fig5_waterfall_lna_quiet.png                               498 KB
  fig5_waterfall_lna_rfi.png                                 436 KB
  fig5_waterfall_nolna_full.png                              673 KB
  fig5_waterfall_nolna_quiet.png                             504 KB
  fig5_waterfall_nolna_rfi.png                               443 KB
  fig5_waterfall_yagi_full.png                               646 KB
  fig5_waterfall_yagi_quiet.png                              495 KB
  fig5_waterfall_yagi_rfi.png                                409 KB
  fig6_hires_spectrum.png                                    602 KB
 

In [20]:
# ================================================================
# CELL A — Normalised power ratio waterfall (Jordan's suggestion)
# Each spectrum normalised to its total in-band power before ratio.
# Right panel: 10*log10(FFT_norm / PFB_norm)
#   Red  = FFT has proportionally MORE power (leakage in)
#   Blue = PFB has more power (carrier correctly localised)
# ================================================================
FRAME_CADENCE_S = 10.0  # seconds per waterfall frame

def normalised_ratio_waterfall(ds, label, out_path):
    wf_fft = ds['wf_fft'].astype(float)   # (360, 93)
    wf_pfb = ds['wf_pfb'].astype(float)
    t_axis = ds['wf_times']               # (360,) actual timestamps or offsets

    # Reconstruct the frequency axis for the 93 science-band channels
    lo_c   = np.searchsorted(ds['freq_c'], RHINO_LO)
    wf_n   = wf_fft.shape[1]
    f_band = ds['freq_c'][lo_c:lo_c + wf_n]

    # Convert dB → linear, normalise each frame to total in-band power
    fft_lin  = 10**(wf_fft / 10.0)
    pfb_lin  = 10**(wf_pfb / 10.0)
    fft_norm = fft_lin  / fft_lin.sum(axis=1, keepdims=True)
    pfb_norm = pfb_lin  / pfb_lin.sum(axis=1, keepdims=True)
    ratio_db = 10 * np.log10(
        np.clip(fft_norm, 1e-30, None) / np.clip(pfb_norm, 1e-30, None)
    )

    ext = [f_band[0], f_band[-1], t_axis[-1], t_axis[0]]

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    for ax, data, title, cmap, sym in [
        (axes[0], wf_fft,  'FFT (Hann)',              'viridis', False),
        (axes[1], wf_pfb,  'PFB (4 taps)',             'viridis', False),
        (axes[2], ratio_db,'FFT/PFB normalised ratio\n'
                           '+red=FFT leaking  -blue=PFB localising',
                            'RdBu_r', True),
    ]:
        if sym:
            lim = np.nanpercentile(np.abs(data), 98)
            vmin, vmax = -lim, lim
        else:
            vmin = np.nanpercentile(data, 2)
            vmax = np.nanpercentile(data, 98)
        im = ax.imshow(data, aspect='auto', origin='upper',
                       extent=ext, vmin=vmin, vmax=vmax, cmap=cmap)
        ax.set_xlabel('Frequency (MHz)'); ax.set_ylabel('Time (s)')
        ax.set_title(title, fontsize=9)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='dB')

    fig.suptitle(f'Normalised power ratio | {label} | {RHINO_LO}-{RHINO_HI} MHz\n'
                 'Each spectrum normalised to total in-band power before ratio',
                 fontsize=10)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {out_path}')

normalised_ratio_waterfall(ds_lna, ds_lna['label'],
                           OUT + '/fig5_normalised_ratio_lna.png')

Saved /Users/user/Downloads/Manny-Masters/Project/Data/rhino_thesis_figs_final/fig5_normalised_ratio_lna.png


In [22]:
# ================================================================
# CELL B — Wider band leakage: 55-120 MHz static spectrum
# Uses fft_c / pfb_c (full 8193-channel single spectra).
# The waterfall is pre-sliced to 60-85 MHz so a full wideband
# waterfall would need re-acquisition; this static comparison
# directly answers Jordan's out-of-band leakage question.
# Key region: 82-88 MHz — FM carriers at 87.5+ MHz leak left into
# the science band in the FFT (-6 dB at +/-1 bin) but are
# suppressed ~38 dB more in the PFB.
# ================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

datasets_wb = [
    (axes[0], ds_lna,   'Discone + ZKL-2+ LNA'),
    (axes[1], ds_yagi,  'Yagi + CobraX LNA'),
]

for ax, ds, title in datasets_wb:
    freq = ds['freq_c']
    fft  = ds['fft_c'].astype(float)
    pfb  = ds['pfb_c'].astype(float)

    mask = (freq >= 55) & (freq <= 120)
    f    = freq[mask]

    ax.plot(f, fft[mask], color=COL['fft'], lw=1.0, alpha=0.85,
            label='FFT (Hann)')
    ax.plot(f, pfb[mask], color=COL['pfb'], lw=1.0, alpha=0.85,
            label='PFB (4 taps)')

    # Shade science band and FM band
    ax.axvspan(RHINO_LO, RHINO_HI, alpha=0.10, color='gold',
               label='Science band 60-85 MHz')
    ax.axvspan(87.5, 108, alpha=0.10, color='salmon',
               label='FM band 87.5-108 MHz')

    # Mark the boundary
    ax.axvline(RHINO_HI, color='goldenrod', lw=0.9, ls='--', alpha=0.8)
    ax.axvline(87.5,     color='salmon',    lw=0.9, ls='--', alpha=0.8)

    # Zoom inset: 80-92 MHz — the critical leakage transition zone
    ax_in = ax.inset_axes([0.55, 0.55, 0.43, 0.40])
    z_mask = (freq >= 80) & (freq <= 92)
    ax_in.plot(freq[z_mask], fft[z_mask],
               color=COL['fft'], lw=1.2, label='FFT')
    ax_in.plot(freq[z_mask], pfb[z_mask],
               color=COL['pfb'], lw=1.2, label='PFB')
    ax_in.axvspan(RHINO_LO, RHINO_HI, alpha=0.12, color='gold')
    ax_in.axvspan(87.5, 92, alpha=0.12, color='salmon')
    ax_in.axvline(87.5, color='salmon', lw=0.8, ls='--')
    ax_in.set_title('80-92 MHz zoom', fontsize=7)
    ax_in.tick_params(labelsize=7)
    ax_in.set_xlabel('MHz', fontsize=7)
    ax_in.set_xlim(80, 92)
    ax_in.set_title('Zoom: 80-92 MHz (leakage transition)', fontsize=7)

    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel('Power (dB, arb.)')
    ax.set_xlim(55, 120)
    ax.set_title(title)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(alpha=0.25)

fig.suptitle('Out-of-band leakage test | 55-120 MHz\n'
             'FM block (87.5-108 MHz) sits immediately above science band — '
             'FFT sidelobes (-6 dB at ±1 bin) carry FM power into 86-87 MHz; '
             'PFB suppresses this by ~38 dB',
             fontsize=10)
fig.tight_layout()
fig.savefig(OUT + '/fig5_wideband_leakage_55_120MHz.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Saved fig5_wideband_leakage_55_120MHz.png')
print('Check the inset: FFT should sit higher than PFB in 82-87 MHz '
      '(FM leaking left); they should converge inside the science band.')

Saved fig5_wideband_leakage_55_120MHz.png
Check the inset: FFT should sit higher than PFB in 82-87 MHz (FM leaking left); they should converge inside the science band.


In [19]:
# Quick diagnostic — run this and paste the output
print("Keys in ds_lna:")
for k, v in ds_lna.items():
    if hasattr(v, 'shape'):
        print(f"  '{k}': shape={v.shape} dtype={v.dtype}")
    elif isinstance(v, list):
        print(f"  '{k}': list of {len(v)}")
    elif v is None:
        print(f"  '{k}': None")
    else:
        print(f"  '{k}': {type(v).__name__} = {v}")

Keys in ds_lna:
  'freq_c': shape=(8193,) dtype=float64
  'freq_h': shape=(524289,) dtype=float64
  'fft_c': shape=(8193,) dtype=float32
  'pfb_c': shape=(8193,) dtype=float32
  'fft_h': shape=(524289,) dtype=float32
  'integ_f': shape=(524289,) dtype=float32
  'integ_freq': shape=(524289,) dtype=float64
  'wf_fft': shape=(360, 93) dtype=float32
  'wf_pfb': shape=(360, 93) dtype=float32
  'wf_fft_q': shape=(360, 49) dtype=float32
  'wf_pfb_q': shape=(360, 49) dtype=float32
  'wf_fft_r': shape=(360, 38) dtype=float32
  'wf_pfb_r': shape=(360, 38) dtype=float32
  'wf_hires': shape=(40, 5927) dtype=float32
  'wf_times': shape=(360,) dtype=float64
  'wf_rms': shape=(360,) dtype=float64
  'raw_ts': shape=(1049055,) dtype=float32
  'label': str = Discone + ZKL-2+ LNA
  'snap_Ns': list of 30
  'snaps': list of 30


In [23]:
print(f"raw_ts shape: {ds_lna['raw_ts'].shape}")
print(f"raw_ts dtype: {ds_lna['raw_ts'].dtype}")
print(f"min: {ds_lna['raw_ts'].min():.4f}  max: {ds_lna['raw_ts'].max():.4f}  mean: {ds_lna['raw_ts'].mean():.4f}")
print(f"Expected spectra at N=16384: {ds_lna['raw_ts'].shape[0] // 16384}")
print(f"Expected spectra at N=1048576: {ds_lna['raw_ts'].shape[0] // 1048576}")

raw_ts shape: (1049055,)
raw_ts dtype: float32
min: -4526.0000  max: 4512.0000  mean: -0.0676
Expected spectra at N=16384: 64
Expected spectra at N=1048576: 1


In [24]:
# ================================================================
# CELL — Window function comparison (Jordan's request)
# Reprocesses the same raw_ts IQ samples through three windows:
# Rectangular, Hann (current), and Blackman (lower sidelobes).
# Coarse mode: 64 frames of N=16384 averaged per window.
# ENBW correction applied so noise floors align — shape differences
# are then genuine, not a level artefact of the window choice.
# ================================================================

FS_MHZ   = 4423.680
N_COARSE = 16384

# Windows with their Equivalent Noise Bandwidths
WINDOWS = {
    'Rectangular':  (np.ones(N_COARSE),       1.000),
    'Hann':         (np.hanning(N_COARSE),     1.500),
    'Blackman':     (np.blackman(N_COARSE),    1.727),
}
COLORS = {
    'Rectangular': '#d62728',
    'Hann':        '#1f77b4',
    'Blackman':    '#2ca02c',
}

# Frequency axis for coarse mode
freq_c = np.fft.rfftfreq(N_COARSE) * FS_MHZ    # 0 to ~2212 MHz

def windowed_spectrum(raw, N, window, enbw):
    """
    Split raw into complete blocks of N, apply window, rfft, average
    power spectra, apply ENBW correction so noise floor aligns.
    """
    n_frames = len(raw) // N
    blocks   = raw[:n_frames * N].reshape(n_frames, N).astype(np.float64)
    spec_sum = np.zeros(N // 2 + 1)
    for block in blocks:
        windowed = block * window
        S = np.fft.rfft(windowed)
        spec_sum += np.abs(S)**2
    spec_avg = spec_sum / n_frames
    spec_avg = np.maximum(spec_avg, 1e-30)
    # ENBW correction: add 10*log10(ENBW) so noise floors align across windows
    spec_db  = 10 * np.log10(spec_avg) + 10 * np.log10(enbw)
    return spec_db

datasets = []
for ds, label in [(ds_lna,   'Discone + ZKL-2+ LNA'),
                  (ds_nolna, 'Discone, no LNA'),
                  (ds_yagi,  'Yagi + CobraX LNA')]:
    if ds.get('raw_ts') is not None:
        datasets.append((label, ds['raw_ts']))
    else:
        print(f'WARN: no raw_ts for {label}')

if not datasets:
    print('ERROR: no raw_ts found in any dataset'); 
else:
    fig, axes = plt.subplots(len(datasets), 2,
                             figsize=(13, 4 * len(datasets)))
    if len(datasets) == 1:
        axes = [axes]

    for row, (label, raw) in enumerate(datasets):
        ax_sci  = axes[row][0]   # science band 60-85 MHz
        ax_wide = axes[row][1]   # wider view 55-120 MHz

        for wname, (window, enbw) in WINDOWS.items():
            spec = windowed_spectrum(raw, N_COARSE, window, enbw)
            col  = COLORS[wname]
            lw   = 1.8 if wname == 'Hann' else 1.1
            alpha = 1.0 if wname == 'Hann' else 0.80

            # Science band panel
            m = (freq_c >= 60) & (freq_c <= 85)
            ax_sci.plot(freq_c[m], spec[m],
                        color=col, lw=lw, alpha=alpha, label=wname)

            # Wider band panel
            m2 = (freq_c >= 55) & (freq_c <= 120)
            ax_wide.plot(freq_c[m2], spec[m2],
                         color=col, lw=lw, alpha=alpha, label=wname)

        for ax, title_suffix, lo, hi in [
            (ax_sci,  '60-85 MHz (science band)', 60,  85),
            (ax_wide, '55-120 MHz (incl. FM)',     55, 120),
        ]:
            ax.axvspan(60,   85,  alpha=0.08, color='gold')
            ax.axvspan(87.5, 108, alpha=0.08, color='salmon')
            ax.set_xlim(lo, hi)
            ax.set_xlabel('Frequency (MHz)')
            ax.set_ylabel('Power (dB, ENBW-corrected)')
            ax.set_title(f'{label}\n{title_suffix}', fontsize=9)
            ax.legend(fontsize=8)
            ax.grid(alpha=0.25)

    fig.suptitle(
        'Window function comparison | Same IQ samples, 64 coarse frames averaged\n'
        'ENBW correction applied — shape differences are genuine, not level offsets',
        fontsize=10)
    fig.tight_layout()
    out_path = OUT + '/fig_window_comparison.png'
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {out_path}')
    print()
    print('What to look for:')
    print('  SHAPES OVERLAP  -> spectral structure is real, not window artefact')
    print('  SHAPES DIVERGE  -> some features are window-dependent (leakage)')
    print('  RFI spikes will look narrower in Blackman (lower sidelobes)')
    print('  Blackman main lobe is wider so very closely spaced lines may merge')

Saved /Users/user/Downloads/Manny-Masters/Project/Data/rhino_thesis_figs_final/fig_window_comparison.png

What to look for:
  SHAPES OVERLAP  -> spectral structure is real, not window artefact
  SHAPES DIVERGE  -> some features are window-dependent (leakage)
  RFI spikes will look narrower in Blackman (lower sidelobes)
  Blackman main lobe is wider so very closely spaced lines may merge
